In [0]:
import json
import time
import requests
from azure.eventhub import EventHubProducerClient, EventData

dbutils.widgets.text(
    "URL",
    "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson"
)

USGS_URL = dbutils.widgets.get("URL")

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-connstr"
)

EVENT_HUB_NAME = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-name"
)

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

try:
    seen_ids = set()
    while True:
        data = requests.get(USGS_URL).json()

        batch = producer.create_batch()

        for earthquake in data["features"]:

            earthquake_id = earthquake["id"]

            if earthquake_id in seen_ids:
                continue

            seen_ids.add(earthquake_id)

            batch.add(
                EventData(
                    json.dumps(earthquake)
                )
            )

        producer.send_batch(batch)

        print(f"Sent {len(data['features'])} earthquakes")

        time.sleep(60)

finally:
    producer.close()